#**DEPENDENCY INSTALLATION**

In [6]:
!pip install transformers datasets sentence-transformers scikit-learn accelerate evaluate rouge_score

# **SEMANTIC_COMMENT_CLUSTERING**

In [7]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering

# 1. Initialize the Embedding Model
# We use 'all-distilroberta-v1' as a fast, high-performance alternative to the paper's specific RoBERTa checkpoint
embedder = SentenceTransformer('all-distilroberta-v1')

def cluster_comments(comments, distance_threshold=0.5):
    """
    Clusters a list of comments based on semantic similarity using Agglomerative Clustering.
    """
    # If there are no comments or very few, just return them as a single group to avoid errors
    if not comments:
        return {}
    if len(comments) == 1:
        return {0: comments}

    # A. Encode comments into vectors (embeddings)
    embeddings = embedder.encode(comments)

    # B. Perform Agglomerative Clustering
    # Paper Settings: Average linkage, Cosine distance, Max distance 0.5
    clustering_model = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=distance_threshold,
        metric='cosine',
        linkage='average'
    )

    clustering_model.fit(embeddings)
    cluster_assignment = clustering_model.labels_

    # C. Group comments by their cluster ID
    clustered_comments = {}
    for idx, cluster_id in enumerate(cluster_assignment):
        cluster_id = int(cluster_id) # Convert numpy int to python int for cleanliness
        if cluster_id not in clustered_comments:
            clustered_comments[cluster_id] = []
        clustered_comments[cluster_id].append(comments[idx])

    return clustered_comments

# ---------------------------------------------------------
# 2. LOAD REAL DATA FROM train.json
# ---------------------------------------------------------

# Load the JSON file
with open('/content/drive/MyDrive/data/downloadable_data/raw/train.json', 'r') as f:
    data = json.load(f)

# Let's pick a specific thread to test (e.g., the first thread in the dataset)
# You can change the index [0] to [1], [2], etc. to see different threads
selected_thread = data['threads'][9]

print(f"Processing Thread ID: {selected_thread['submission_id']}")
print(f"Subreddit: {selected_thread['subreddit']}")
print(f"Original Caption: {selected_thread['caption']}\n")

# Extract just the text body of each comment
# The JSON structure is: thread -> 'comments' -> list of dicts -> 'body'
real_comments = [c['body'] for c in selected_thread['comments']]

# Remove deleted or empty comments if necessary
real_comments = [c for c in real_comments if c not in ["[deleted]", "[removed]"] and c.strip() != ""]

print(f"Found {len(real_comments)} valid comments. Clustering now...\n")

# 3. Run the Clustering
op_clarification_text = real_comments[0]
community_comments = real_comments[1:]
clusters = cluster_comments(community_comments)

# 4. Display Results
print("-" * 30)
for cluster_id, comment_list in clusters.items():
    print(f"CLUSTER {cluster_id} ({len(comment_list)} comments):")
    for comment in comment_list:
        # Print first 100 chars to keep output readable
        preview = comment[:100] + "..." if len(comment) > 100 else comment
        print(f"  - {preview}")
    print("-" * 30)

Processing Thread ID: bgp7dn
Subreddit: malelivingspace
Original Caption: living room in my condo. any advice on wall art or a splash of color to make it feel warmer?

Found 7 valid comments. Clustering now...

------------------------------
CLUSTER 5 (1 comments):
  - Some sheer curtains under the existing curtains usually look pretty nice. Also, a couple plants woul...
------------------------------
CLUSTER 4 (1 comments):
  - Tuscan yellow (marigold, mustard, honey) accents (like sofa pillows), they pull up the blah gray and...
------------------------------
CLUSTER 3 (1 comments):
  - ID on that rug?
------------------------------
CLUSTER 2 (1 comments):
  - Since you like maps there are some colorful options out there. Instead of that severe and modern map...
------------------------------
CLUSTER 1 (1 comments):
  - You got a nice wall art but a quite small for your wide wall ! I think you need to move the lamp nea...
------------------------------
CLUSTER 0 (1 comments):
  - Too

# **SUPERVISED_FINE_TUNING_ON_MREDDITSUM**

In [11]:

import torch
from datasets import Dataset, DatasetDict
from transformers import BartTokenizer, BartForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
import evaluate
import numpy as np

# 1. Load Data from Text Files
def load_data_from_text(src_file, tgt_file):
    with open(src_file, 'r', encoding='utf-8') as f:
        sources = [line.strip() for line in f.readlines()]
    with open(tgt_file, 'r', encoding='utf-8') as f:
        targets = [line.strip() for line in f.readlines()]
    return Dataset.from_dict({"text": sources, "summary": targets})

# 2. LOAD ALL THREE DATASETS (Train: 2729, Val: 152, Test: 152)
# Ensure these paths correctly point to your Drive files
base_path = '/content/drive/MyDrive/data/downloadable_data/preprocessed/'

train_dataset = load_data_from_text(f'{base_path}train_processed_imgcap_src.txt', f'{base_path}train_processed_imgcap_tgt.txt')
val_dataset = load_data_from_text(f'{base_path}val_processed_imgcap_src.txt', f'{base_path}val_processed_imgcap_tgt.txt')
test_dataset = load_data_from_text(f'{base_path}test_processed_bestimgcap_src.txt', f'{base_path}test_processed_bestimgcap_tgt.txt')

# Combine into a proper DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})

# 3. Tokenizer & Model Setup
model_checkpoint = "facebook/bart-base"
tokenizer = BartTokenizer.from_pretrained(model_checkpoint)
model = BartForConditionalGeneration.from_pretrained(model_checkpoint)

# 4. Preprocessing
max_input_length = 512
max_target_length = 128

def preprocess_function(examples):
    inputs = ["summarize: " + doc for doc in examples["text"]]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)
    labels = tokenizer(text_target=examples["summary"], max_length=max_target_length, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

# 5. Metrics
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}

# 6. Updated Training Arguments (paper's has 50 epochs and 3e-5 learning rate)
args = Seq2SeqTrainingArguments(
    output_dir="./t5-mredditsum-final-splits",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,          # Paper's recommended rate [cite: 719]
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=20,         # Paper trained for 50 epochs
    predict_with_generate=True,
    fp16=True,
    load_best_model_at_end=True, # Critical: Uses validation set to pick best model
    metric_for_best_model="rouge1",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"], # Training monitors validation performance
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    compute_metrics=compute_metrics,
)

# 7. Train
trainer.train()

# 8. FINAL EVALUATION ON THE TEST SET
print("\n--- Final Evaluation on Unseen Test Set ---")
test_results = trainer.evaluate(eval_dataset=tokenized_datasets["test"])
print(f"Test ROUGE Scores: {test_results}")

# 9. Save
trainer.save_model("./final_mredditsum_model_bart_base_with_img_caption_20_epoch")

Map:   0%|          | 0/2729 [00:00<?, ? examples/s]

Map:   0%|          | 0/152 [00:00<?, ? examples/s]

Map:   0%|          | 0/152 [00:00<?, ? examples/s]

/tmp/ipython-input-1854756876.py:76: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,2.436200,2.071946,21.528200,11.869200,18.917500,18.902900
2,2.169700,2.012495,21.811500,11.841700,19.068300,19.067500
3,1.919500,1.959469,21.472000,11.701000,18.763000,18.763700
4,1.812400,1.979119,22.302800,12.318300,19.541800,19.516000
5,1.753900,1.971385,21.957400,12.412200,19.362000,19.339500
6,1.617300,1.981330,21.944200,12.141000,19.190300,19.173700
7,1.555600,2.005376,21.587600,11.934900,18.866200,18.860900
8,1.514200,2.019416,21.772800,12.121300,19.140200,19.131000
9,1.403000,2.024713,21.990700,12.292500,19.246900,19.259800
10,1.375900,2.064471,21.811400,12.266200,19.213300,19.233800


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].



--- Final Evaluation on Unseen Test Set ---


Test ROUGE Scores: {'eval_loss': 2.0226879119873047, 'eval_rouge1': 22.1357, 'eval_rouge2': 12.4526, 'eval_rougeL': 19.4609, 'eval_rougeLsum': 19.498, 'eval_runtime': 12.9844, 'eval_samples_per_second': 11.706, 'eval_steps_per_second': 2.927, 'epoch': 20.0}


# **SINGLE_PASS_BASELINE_INFERENCE**

In [12]:
from transformers import pipeline

# Load your fine-tuned model
summarizer = pipeline("summarization", model="/content/final_mredditsum_model_bart_base_without_img_caption_20_epoch", tokenizer=tokenizer, device=0)

# Example input (You can paste a raw line from your src.txt here)
input_text = "Original Post: What color should I paint my walls? Image: A living room with beige furniture. OP: I feel like it's too boring. User 1: Try sage green! User 2: I agree, green would look great."

# T5 prompt prefix
input_text = "summarize: " + input_text

summary = summarizer(input_text, max_length=128, min_length=30, do_sample=False)
print("Generated Summary:", summary[0]['summary_text'])

Device set to use cuda:0
Your max_length is set to 128, but your input_length is only 55. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=27)
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated Summary: op wanted advice on what color they should paint the walls in their living room, which is painted beige. The OP added that they wanted to paint their walls a sage green color. One commenter suggested sage green for the walls.


# **CMS_PIPELINE_EXECUTION_(SUMMARIZATION_&_SYNTHESIS)**

In [13]:
import torch

# ==========================================
# HELPER: SINGLE SUMMARY GENERATION
# ==========================================
def summarize_text(text, model, tokenizer, max_length=128):
    """
    Runs the T5 model on a single chunk of text.
    """
    # Prepare input with the T5 prefix
    input_text = "summarize: " + text

    # Tokenize and move to GPU
    device = model.device
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)

    # Generate summary
    # Beam search of 5 is used in the paper (Section 4.5 Implementation Details)
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=max_length,
        min_length=10,
        length_penalty=2.0,
        num_beams=5,
        early_stopping=True,
        no_repeat_ngram_size=3
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# ==========================================
# STAGE 2: CLUSTER SUMMARIZATION
# ==========================================
def stage_2_summarize_clusters_fixed(clustered_comments, op_caption, op_text, model, tokenizer):
    print(f"Stage 2: Summarizing OP (Caption + Text) and {len(clustered_comments)} comment clusters...")

    # 1. Summarize the Original Post (OP) with the caption AND the clarifying text
    # This provides the T5 model with the full context of the user's problem.
    op_input = f"Original Post: {op_text} Image: {op_caption}"
    op_summary = summarize_text(op_input, model, tokenizer, max_length=150) # Increased max_length

    cluster_summaries = []

    # 2. Summarize each cluster individually
    for cluster_id, comments in clustered_comments.items():
        cluster_text = " ".join([f"Commenter: {c}" for c in comments])
        c_summary = summarize_text(cluster_text, model, tokenizer)
        cluster_summaries.append(c_summary)

    return op_summary, cluster_summaries

# ==========================================
# STAGE 3: SYNTHESIS (FINAL SUMMARY)
# ==========================================
def stage_3_synthesis(op_summary, cluster_summaries, model, tokenizer):
    print("Stage 3: Synthesizing final summary...")

    # Concatenate the OP summary and all Cluster summaries
    # This matches the "Cluster-summary Summarization" step in Figure 1 of the paper
    combined_text = f"Post Summary: {op_summary} Comment Summaries: " + " ".join(cluster_summaries)

    # Run the model one last time on this combined text
    final_summary = summarize_text(combined_text, model, tokenizer, max_length=256)

    return final_summary

# ==========================================
# MAIN EXECUTION (Using variables from Cell 1 & 2)
# ==========================================
op_clarification_text = real_comments[0]
community_comments = real_comments[1:]
op_caption = selected_thread['caption']


print(f"Running CMS Pipeline on Thread: {selected_thread['submission_id']}")
print(f"OP Clarification Text separated: '{op_clarification_text[:50]}...'")

op_summary_fixed, cluster_summaries_fixed = stage_2_summarize_clusters_fixed(
    clusters,
    op_caption,
    op_clarification_text,
    model,
    tokenizer
)

print("\n--- Intermediate Results (Fixed) ---")
print(f"OP Summary (Fixed): {op_summary_fixed}")
for i, c_sum in enumerate(cluster_summaries_fixed):
    print(f"Cluster {i} Summary: {c_sum}")

# Re-run Stage 3 Synthesis
final_cms_output_fixed = stage_3_synthesis(op_summary_fixed, cluster_summaries_fixed, model, tokenizer)

print("\n" + "="*40)
print("FINAL CMS GENERATED SUMMARY (FIXED)")
print("="*40)
print(final_cms_output_fixed)

Running CMS Pipeline on Thread: bgp7dn
OP Clarification Text separated: 'Some plants (fake/real) might liven it up and look...'
Stage 2: Summarizing OP (Caption + Text) and 6 comment clusters...

--- Intermediate Results (Fixed) ---
OP Summary (Fixed): oceanize The OP asked for some plants to liven up their living room, which has white walls and a tan sofa.  One commenter suggested adding some wall art or a splash of color to make it feel warmer.  Another commenter recommended adding plants to the room.
Cluster 0 Summary: commenter: Some sheer curtains under the existing curtains would look nice. A couple plants would help pull it together.
Cluster 1 Summary: commenter: Tuscan yellow accents are not girlish and do not pull up the gray in the room. One commenter recommended a large canvas of a sunrise on the wall behind the sofa. Another commenter recommended split the canvas into pieces, vertical or horizontal.
Cluster 2 Summary: commenter: The OP wanted to know the ID on the rug that 